# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [7]:
# imports
import os
from dotenv import load_dotenv
from openai import OpenAI, APIError, APIConnectionError
from IPython.display import Markdown, display, update_display
from typing import Optional, Union, Dict, Any, Tuple

# IMPROVEMENT: Added type hints imports for better code documentation and IDE support
# IMPROVEMENT: Added specific OpenAI exception types for better error handling

In [8]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# IMPROVEMENT: Extracted magic strings to constants for better maintainability
# This makes it easier to change display formatting in one place
SEPARATOR_LENGTH = 60  # Length of separator line for output formatting

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")


openai = OpenAI()

API key looks good so far


In [9]:
# set up environment

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [10]:
# here is the question; type over this to ask something new

#question = """
#Please explain what this code does and why:
#yield from {book.get("author") for book in books if book.get("author")}
#"""

code_snippet = """
yield from {book.get("author") for book in books if book.get("author")}
"""

# IMPROVEMENT: Added type hints - makes function signature self-documenting
# IMPROVEMENT: Added input validation - fails fast with clear error messages
def create_learning_question(code_snippet: str) -> Tuple[str, str]:
    """
    Create system and user prompts for code explanation.
    
    Args:
        code_snippet: Code to explain (must not be empty)
    
    Returns:
        Tuple of (system_message, user_message)
    
    Raises:
        ValueError: If code_snippet is empty or whitespace only
    """
    # IMPROVEMENT: Input validation - catch errors early with clear messages
    # This prevents API calls with invalid data and saves costs
    if not code_snippet or not code_snippet.strip():
        raise ValueError("code_snippet cannot be empty or whitespace only")
    
    system_message = """
    You are a patient, educational coding mentor helping someone learn programming.
    Your goal is to explain code in a way that helps them understand not just what
    the code does, but how to recognize and apply similar patterns in the future.
    
    Always:
    - Break things down step-by-step
    - Explain the "why" behind approaches, not just the "what"
    - Compare to similar patterns when relevant
    - Be clear and thorough
    """



    user_message = f"""
    Please explain this code step-by-step in a way that helps me learn:

    {code_snippet}

    Structure your explanation as:
    1. **High-level overview**: What problem does this solve?
    2. **Step-by-step breakdown**: Walk through each part of the code
    3. **Key concepts**: What programming patterns/techniques are used here?
    4. **Why this approach**: What are the tradeoffs vs alternative approaches?
    5. **Similar patterns**: How does this compare to other approaches I might have seen?
    6. **Practical application**: When would I use similar patterns in my own code?
    7. **Watch out for**: Any potential gotchas or edge cases?

    Make it educational - help me understand how to recognize and apply similar patterns in future code.
    """

    return system_message, user_message

In [11]:
# Code explanation tool: Unified interface for any OpenAI-compatible model
#
# IMPROVEMENTS IMPLEMENTED (v2):
# 1. Type hints - Better IDE support, self-documenting code, catch errors early
# 2. Input validation - Fail fast with clear error messages, prevent invalid API calls
# 3. Specific error handling - Different exceptions handled appropriately with helpful hints
# 4. Performance optimization - List-based string concatenation for streaming (O(n) vs O(n²))
# 5. Edge case handling - Empty response detection prevents downstream errors
# 6. Enhanced docstrings - More detailed documentation with Raises section
# 7. Constants extraction - Magic numbers/strings extracted for maintainability
# 8. Better error messages - Include exception types and actionable hints

# IMPROVEMENT: Added comprehensive type hints for better IDE support and documentation
# IMPROVEMENT: More specific return type annotation (Union handles multiple return types)
def ask_about_code(
    code_snippet: str,
    model_client: OpenAI,
    model_name: str,
    stream: bool = False,
    should_display: bool = True,
    return_full: bool = False
) -> Union[str, Dict[str, Any], Any, None]:
    """
    Get explanation of code from any OpenAI-compatible model.
    
    IMPROVEMENT: Enhanced docstring with more detail and Raises section
    
    Args:
        code_snippet: Code to explain (validated to be non-empty)
        model_client: OpenAI client instance (configured for OpenAI or Ollama)
        model_name: Model name string (e.g., 'gpt-4o-mini' or 'llama3.2')
        stream: Whether to stream response (real-time display)
        should_display: Whether to display result in notebook (Markdown rendering)
        return_full: Return full response object with metadata vs just text
    
    Returns:
        - If return_full=False: Response text (str)
        - If return_full=True and stream=True: Dict with text, model, streaming flag
        - If return_full=True and stream=False: Full response object with metadata
        - None if error occurs
    
    Raises:
        ValueError: If code_snippet is empty (from create_learning_question)
        APIConnectionError: If connection to API fails
        APIError: If API returns an error
    """
    # IMPROVEMENT: Input validation happens in create_learning_question
    # This will raise ValueError if code_snippet is invalid
    system_msg, user_msg = create_learning_question(code_snippet)
    
    # IMPROVEMENT: More specific exception handling - different errors need different handling
    # This provides better debugging information and user feedback
    try:
        if stream:
            # IMPROVEMENT: Use list for string concatenation - more efficient than += in loops
            # String concatenation with += creates new string objects each time (O(n²) complexity)
            # Using list and join is O(n) - important for long streaming responses
            collected_chunks = []
            display_handle = display(Markdown(""), display_id=True) if should_display else None
            
            stream_response = model_client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": user_msg}
                ],
                stream=True
            )
            
            for chunk in stream_response:
                chunk_text = chunk.choices[0].delta.content or ''
                if chunk_text:  # IMPROVEMENT: Only append non-empty chunks
                    collected_chunks.append(chunk_text)
                    if should_display and display_handle:
                        # IMPROVEMENT: Join accumulated chunks for display (more efficient)
                        current_text = ''.join(collected_chunks)
                        update_display(Markdown(current_text), display_id=display_handle.display_id)
            
            # IMPROVEMENT: Join once at the end instead of concatenating in loop
            collected_text = ''.join(collected_chunks)
            
            # IMPROVEMENT: Handle empty response edge case
            if not collected_text:
                print(f"Warning: {model_name} returned empty response")
                return None
            
            if return_full:
                return {
                    "text": collected_text,
                    "model": model_name,
                    "streaming": True
                }
            else:
                return collected_text
        else:
            response = model_client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": user_msg}
                ]
            )
            
            result_text = response.choices[0].message.content
            
            # IMPROVEMENT: Handle empty response edge case - prevents downstream errors
            if not result_text:
                print(f"Warning: {model_name} returned empty response")
                return None
            
            if should_display:
                display(Markdown(result_text))
            
            if return_full:
                return response
            else:
                return result_text
                
    # IMPROVEMENT: Specific exception handling - provides better error context
    # APIConnectionError: Network/connection issues (retry might help)
    # APIError: API-level errors (auth, rate limits, etc.)
    # Generic Exception: Catch-all for unexpected errors
    except APIConnectionError as e:
        print(f"Connection error with {model_name}: {e}")
        print("Hint: Check your internet connection or API endpoint")
        return None
    except APIError as e:
        print(f"API error with {model_name}: {e}")
        print("Hint: Check your API key, rate limits, or model availability")
        return None
    except Exception as e:
        # IMPROVEMENT: More informative error message with exception type
        print(f"Unexpected error calling {model_name}: {type(e).__name__}: {e}")
        return None

# IMPROVEMENT: Added type hints for better code documentation
def compare_explanations(
    code_snippet: str,
    gpt_client: OpenAI,
    llama_client: OpenAI
) -> Tuple[Optional[str], Optional[str]]:
    """
    Get explanations from both models and compare side-by-side.
    
    IMPROVEMENT: Enhanced docstring with more detail
    
    Args:
        code_snippet: Code to explain
        gpt_client: OpenAI client configured for GPT models
        llama_client: OpenAI client configured for Ollama/Llama models
    
    Returns:
        Tuple of (gpt_result, llama_result) - either can be None if error occurs
    """
    # IMPROVEMENT: Use constant instead of magic number - easier to maintain
    separator = "=" * SEPARATOR_LENGTH
    
    print(separator)
    print(f"{MODEL_GPT} Explanation:")
    print(separator)
    gpt_result = ask_about_code(code_snippet, gpt_client, MODEL_GPT, stream=False, should_display=True)
    
    print("\n" + separator)
    print(f"{MODEL_LLAMA} Explanation:")
    print(separator)
    llama_result = ask_about_code(code_snippet, llama_client, MODEL_LLAMA, stream=False, should_display=True)
    
    return gpt_result, llama_result



In [12]:
# Testing and usage examples

#result = ask_about_code(code_snippet, openai, MODEL_GPT, stream=True)


#full_response = ask_about_code(code_snippet, openai, MODEL_GPT, return_full=True)
#print(f"Tokens: {full_response.usage.total_tokens}")


gpt_result, llama_result = compare_explanations(code_snippet, openai, ollama)

gpt-4o-mini Explanation:


Sure! Let's dive into this code step by step.

### 1. **High-level overview**:
The code you presented is aimed at extracting unique authors from a list of books. Each book is expected to have a key called "author". The goal is to gather and yield these authors, but only if they have a valid value (i.e., not `None` or an empty string).

### 2. **Step-by-step breakdown**:
Let's dissect the code snippet:

```python
yield from {book.get("author") for book in books if book.get("author")}
```

- **`for book in books`**: 
  This part iterates over a collection called `books`. Each `book` represents an individual item in that collection.

- **`book.get("author")`**:
  This takes the current `book` and attempts to retrieve the value associated with the key `"author"`. The `get` method is used instead of dictionary indexing (like `book["author"]`) because `get` will return `None` if the key doesn't exist, instead of raising an exception.

- **`if book.get("author")`**:
  This is a filter that ensures only books with a valid author (i.e., the author is not `None` or a falsy value) are considered for creating the set.

- **`{ ... for book in books if book.get("author") }`**:
  This entire construct is called a set comprehension. It constructs a set, which inherently ensures that all entries are unique. As it iterates through each book, it adds only the author values where the condition (`if book.get("author")`) is true.

- **`yield from`**:
  This keyword is used in generator functions in Python. It allows you to yield all of the values from an iterable (in this case, the set of authors). Using `yield` lets you produce values one at a time, which can be more memory-efficient than returning all values at once.

### 3. **Key concepts**:
- **Set comprehension**: A concise way to create a set by filtering and transforming its elements.
- **Generator functions**: These use `yield` to return values one by one, preventing high memory usage, especially with large datasets.
- **Safe dictionary access**: Using `get()` prevents errors that may arise from trying to access keys that don’t exist.

### 4. **Why this approach**:
- **Efficiency**: By using a set comprehension, we ensure that duplicates are automatically removed, making it an efficient way to get unique authors.
- **Readability**: The use of comprehensions makes the code succinct and easy to read, reducing boilerplate code.
- **Memory Considerations**: Using `yield from` is beneficial when processing large datasets because it allows one author to be processed at a time instead of loading all authors into memory at once.

**Alternative approaches** might include a traditional loop to collect authors in a list and then converting that list to a set for uniqueness. While that works, it is less concise and less efficient in terms of memory and performance.

### 5. **Similar patterns**:
You might have also seen:
- **List comprehensions**: They're almost identical but create lists instead of sets. Example:
  ```python
  [book.get("author") for book in books if book.get("author")]
  ```

- **Generator expressions**: These provide a memory-efficient way of iterating over items similar to list comprehensions but use parentheses instead of square brackets.

### 6. **Practical application**:
You would use this pattern when dealing with datasets from APIs, databases, or any collections where you need to filter out unique items based on a specific attribute (like authors, names, IDs), especially if the dataset might be large. 

### 7. **Watch out for**:
- **Empty dataset**: If `books` is an empty list, the result will also be empty, which is expected, but it’s worth being aware of in case you're not handling empty outputs elsewhere.
- **Types of values**: Make sure that the values associated with "author" are of the type you expect (strings). If you were expecting strings but got other types (e.g., numbers or lists), make sure to handle those cases appropriately.

By understanding this pattern, you'll be better equipped to recognize similar structures in future coding tasks, whether it's for extracting unique values, filtering data, or utilizing comprehensions and generator functions effectively.


llama3.2 Explanation:


I'd be happy to break down this code step-by-step.

**1. High-level overview**

This code solves the problem of extracting a list of authors from a collection of books. Specifically, it iterates over a list of book objects (`books`), filters out any books that don't have an author, and yields each author's name.

**2. Step-by-step breakdown**

Here's what happens in this code:

- `{book.get("author") for book in books if book.get("author")}`: This part creates a generator expression that:
  - Iterates over each book object (`book`) in the `books` collection.
  - Filters out any books that don't have an author using the `if` condition. In Python, you can access attributes of objects (like "author") after they've been checked for existence (`book.get("author}`). If a book doesn't have an attribute, this will return `None`, so in most cases this checks will filter out books without authors.
  - Extracts the value associated with the "author" key from each `book` object using the `.get()` method. 

- `yield from`: This is a syntax for combining generators with function calls. It allows us to take output from another generator and use it as output.

So, in effect, this code takes all these books, filters out those that don't have authors, extracts each author's name from the filtered books, and delivers them one by one.

**3. Key concepts**

This code uses a few key concepts:

* Generator expressions (`{expression for variable}`): create generators without ever using `yield` inside.
* List comprehensions (although not typically used in this way, there is some syntax here that looks similar to list comprehensions)
* `.get()` method: allows you to safely access attributes of objects while considering cases where an attribute does or doesn't exist.

**4. Why this approach?**

This approach is chosen for a few reasons:

- It is efficient because it doesn't iterate over all the books at once, but only iterates over those that do have authors.
 
And also because we use generator expressions which are more memory-effective compared to having them as regular functions.

**5. Similar patterns**

If you've seen code using similar patterns for iterating over collections and filtering certain elements, this pattern looks a bit like a combination of list comprehensions in Python (although that's not exactly used here) with regular `for` loops or generators.

**6. Practical application**

This kind of code should be familiar to any Python programmer because it appears in various contexts where we want to deal with data:

- We've used this in libraries, particularly when working with databases.
 
In particular, it is common when you're performing an API fetch with `requests` library:
```python
import requests

def get_authors():
    # Fetching books from the database using REST API
    response = requests.get("https://db.com/api/books")
    
    # If there's some error in fetching or parsing the books, then we'll catch it.
    data = response.json() if 'data' in response else []
}

# We can iterate over the extracted authors list.
for author in get_authors():
    print(author)
```

For web scraping with BeautifulSoup or similar libraries you'd probably do something very similar.

**7. Watch out for**

One thing to be aware of is `None` values being returned from `.get()`. Although Python handles this well, when working iteratively (like in these loops and function calls), it's worth checking if there are any `None` values to avoid issues.